In [1]:
from dataloader.Dataloader import *
from model.Classifier import *

In [2]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model = Classifier(out_dim=8).to(device) 
train_tsv_file = "data/CrisisMMD_v2.0/crisismmd_datasplit_all/task_humanitarian_text_img_train.tsv"
img_dir = "data/CrisisMMD_v2.0"
train_dataset = CrisisDataset(train_tsv_file, "data/CrisisMMD_v2.0", model.clip_preprocess)

In [3]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=2
)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.head.parameters(), lr=1e-3)

n_epochs = 20


In [ ]:
for epoch in range(10):
    model.train()
    total_loss = 0.0
    correct = 0
    total = 0

    for batch in train_loader:
        images = batch["image"]
        texts = batch["text"]
        labels = batch["label"].to(model.device)

        logits = model(images, texts)
        loss = criterion(logits, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * labels.size(0)
        preds = logits.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    print(f"Epoch {epoch+1}: loss={total_loss/total:.4f}, acc={correct/total:.4f}")